# Simulating Planetary Orbits 

Within this notebook for my final assignment, I have been asked to simulate planetary orbits via Euler's Method of approximation. This notebook will be split up into parts: an Introduction, in which the maths used will be outlined, followed by orbits of a singular planet, then by two interacting objects and finishing with further investgation conducted by me. 

## Introduction

Within this section we will outline the maths and how the derivations will be useful to our understanding of what will be coded within this notebook. 

In essence, the maths behind orbiting systems relies fundamentally om Newtons law of Gravity which is: 
$$
F = G \frac{Mm}{r^2}
$$

Where G is the fundamental constant of gravity, The M and m being the masses of the two interacting objects and r being the distance between the two objects. Therefore this will be used to find the force acting between the two objects and how it will change depending on the masses of them as well as the distance between them.

However this alone wont be useful for the code as we would need to convert them to numerical model so as to be able to update the position of the planets in the animations. To do this we first must assume that the star is within a fixed position and that the acceleration acting on the star due to the planet is neglible. This will in turn ensure that the star is in a fixed position. Secondly we will need to introduce a time step for Euler's numerical method of approximation to work. It will end up looking like: 
$$
F = m \frac{dv}{dt} \approx m \frac{\delta v}{\delta t}
$$

Rearranging this will give us the final equations:
$$\delta r = v \delta t$$

$$v(t + \delta t) = v(t) - \frac{GM \hat{r}}{\lvert r \rvert^2} \delta t$$

These will be used throughout the notebook.

## Section 1: A Single Planet 

Within this section I will be using functions which I have already coded in the previous task. 

### Main Functions and Data

In [ ]:
## Import the main libraries to be used
import matplotlib.pyplot as plt
import numpy as np

## This will be used to create the animations
from phas0007anim_v2 import animate_trajectory_2d, animate_trajectories_2d

In [ ]:
## This will be used as constants 
G       = 6.6743e-11 # gravitational constant [m^3 kg^-1 s^-2]
YEAR    = 3.154e7    # one year in seconds
M_SOLAR = 1.989e30   # solar mass [kg]
AU      = 1.496e11   # one astronomical unit (mean orbital radius of Earth) [m]

Below I have the main components of this notebook. They have been taken from my previous assignment.

In [ ]:
def force(pos1, pos2, m1, m2):
    """
    Calculate the gravitational force exerted by object 1 on object 2.
    
    Input:
        pos1: position vector of first object  (NumPy array)
        pos2: position vector of second object (NumPy array)
        m1:   mass of first object
        m2:   mass of second object

    Depends on:
        G  (global variable):    gravitational constant [m^3 kg^-1 s^-2]

    Return:
        force on object 2 [N] (NumPy array)
    """
    r_vec = pos1 - pos2
    d = np.linalg.norm(r_vec)

    unit_vec = r_vec / d

    F = ( (G * m1 * m2) / (d**2) ) * unit_vec

    return F
    
def move_planet(position, velocity, m_star, dt):
    """
    Calculate motion of planet in the gravitational field of a star with given mass
    at the originin two dimensions, using Euler's method.
    
    Input:
      - position: position vector of planet at start of time step (NumPy array: x, y)
      - velocity: velocity vector of planet at start of time step (NumPy array: vx, vy)
      - m_star:   mass of star
      - dt:       time step
      
    Output: (position_new, velocity_new)
      - position_new: position vector of planet at end of time step (NumPy array: x, y)
      - velocity_new: velocity vector of planet at end of time step (NumPy array: vx, vy)
      
    Depends on:
      - force = function to calculate the gravitational force between two objects
    """
    pos_star = np.array([0.0, 0.0])
    position_vec = position - pos_star

    d = np.linalg.norm(position_vec)
    unit_vec = position / d

    delta_v = ( (G * m_star) / (d **2 ) ) * unit_vec
    
    position_new = position + velocity * dt
    velocity_new = velocity - delta_v * dt

    return position_new, velocity_new

def trajectory(position, velocity, m_star, dt, t_max):
    """
    Calculate trajectory of planet from given starting position in two dimensions, using Euler's method.
    
    Input:
      - position: position vector of planet at start of simulation [m] (NumPy array: x, y)
      - velocity: velocity vector of planet at start of simulation [m] (NumPy array: vx, vy)
      - m_star:   mass of star [kg]
      - dt:       time step    [s]
      - t_max:    duration of calculated motion [s]

    Returns:
        Trajectory as a 2 x N array containing x and y coordinates at each instant.
    """
    t = 0 
    x, y = position

    x_arr, y_arr = [x], [y]
    
    while t < t_max:
        t += dt 
        position, velocity = move_planet(position, velocity, m_star, dt)
        x, y = position
        x_arr.append(x)
        y_arr.append(y)

    return np.array([x_arr, y_arr])

def plot_trajectory(x, y, label="trajectory", fmt='-'):
    """Plot trajectory with equal aspect ratio, using existing figure.

    Arguments:
        x, y:  arrays of x- and y-coordinates
        label: (optional) label for use in legend
        fmt:   (optional) format to use for plotting, e.g. ':' for dotted line
    """
    plt.plot(x,y,fmt,label=label)
    plt.gca().set_aspect('equal') # same scale horizontally and vertically
    plt.xlabel('x [m]')
    plt.ylabel('y [m]')

### Simulating 5 Years

To simulate 5 years of an orbit, we will need to edit the t_max variable and make that be worth 5 years so as to see the position of the planet change as time goes on.

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/10000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

## This is used to skip every 200 data points. This makes the simulations faster to compile
animate_trajectory_2d(traj_earth[:,::200],30)

For the final line of the previously written code  - animate_trajectory_2d(traj_earth[:,::200],30), we will take data point every 200 points. This is known as splicing the array and only selecteing every 200th point[1]. This makes it easier for processsing and makes it shorter to process the code. 

### Varying Time Step

Within this section I will be varying the timestep. The timestep is the time that passes between every calculation in the simulation. Varying this may have drastic results as we shall see. We will still use the 5 year simulation so as to view the effects the timestep has on the 5 year run of orbits.

First we shall look at it with the timestep being YEAR/1000

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt_1 = YEAR/1000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt_1,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Time Step = 1000")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

Pretty clearly with a timestep of YEAR/1000, we can see that the orbit is moving further and further away from the orbit that it had earlier. This does show that 1000 as the denominator is not a good baseline as it will continue to spiral away.

Now we will see it with 5000

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt_2 = YEAR/5000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt_2,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Time Step = 5000")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

As seen above the orbit is not yet stable and still continues to spiral away.

Now with 50000

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/50000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Time Step = 50000")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

As seen above, the animation comes through smoothly on when using the timestep to be 50000. However this does make the runtime of the code considerably larger. 

For this reason we will try 25000.

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/25000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Time Step = 25000")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

For this set up as seen above, the timestep best suited for the purpose of these simulations is the YEAR/ 25000 and using the frameskip of 300. This makes it both smooth and accurate for the animation. 

### Varying Initial Speed 

Within this section, I will be varying the initial speed of Earth at the beginning of its orbit and seeing how it affects the simulation thereafter. As we kknow to find the velocity of the planet using Newtons laws using the equation: 
$$
V = \sqrt{\frac{GM}{r}}
$$

This will normally give an average value of 29.78 km/s. We will be changing this and seeing how this affects the simulated orbit of the planet. To start off we will use 15km/s effectively lowering the velocity to half of the original.

In [ ]:
## Calculate orbital
v_earth = 15000

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/25000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Speed = 15 Km/s")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

As seen above, we have changed none of the other parameters and only changed the velocity parameter. This has changed the orbit drastically within the 5 year period with it now showing that the distance from the sun starts to get larger with every pass it makes. It also causes the shape of the orbit to change and become elliptical while simultaneously moving further and further away.

Now we will try increasing the velocity from its base value. We will now change the velocity to 50km/s.

In [ ]:
## Calculate orbital
v_earth = 50000

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/25000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Speed = 50 Km/s")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

The simulation here shows a hyperbolic trajectory with the planet going out of the gravitational field and wont return to the solar system. The reason for this is that the velocity has exceeded the escape velocity of the solar system [2]. Escape Velocity is calculated to be:
$$ 
v_{esc} = v_{cc} \times \sqrt{2}
$$
where $v_{esc}$ is the escape the velocity of a planet and $v_{cc}$ is the circular velocity. For these values:
$$
v_{cc} = 29.8 Km/s
$$
$$
v_{esc} = 42.1Km/s
$$

We will now use this value

In [ ]:
## Calculate orbital
v_earth = 42100

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

## Calculate trajectory
dt = YEAR/25000                
traj_earth = trajectory(pos_earth,vel_earth,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = traj_earth

plt.figure()
plt.title("Earth's orbit - Speed = 42.1 Km/s")
x_earth, y_earth = traj_earth
plot_trajectory(x_earth, y_earth)

## Section 2: Two-Interacting Planets

From the code above, we will need to change it so that two planets can be simulated as having an effect on one another. For that, we will need to change the parameters which are being taken in and change the lines within it to account for the extra planet. 

In [ ]:
def move_planets(position, velocity, mass, m_star, pos_other, m_other, dt):
    """
    The position is updated using the velocity from the START of the timestep.
    """
    pos_star = np.array([0.0, 0.0])
    
    ## Calculate Forces
    f_star = force(pos_star, position, m_star, mass)
    f_planet = force(pos_other, position, m_other, mass)
    
    ## Total acceleration (a = F / m)
    acceleration = (f_star + f_planet) / mass
    
    position_new = position + velocity * dt
    velocity_new = velocity + acceleration * dt

    return position_new, velocity_new

def trajectories(pos1, vel1, m1, pos2, vel2, m2, m_star, dt, t_max):
    """
    Calculates trajectories for two interacting planets.
    """
    t = 0 
    ## Initialize storage for both planets
    x1_arr, y1_arr = [pos1[0]], [pos1[1]]
    x2_arr, y2_arr = [pos2[0]], [pos2[1]]
    
    while t < t_max:
        ## Move Planet 1 (influenced by Star and Planet 2)
        p1_new, v1_new = move_planets(pos1, vel1, m1, m_star, pos2, m2, dt)
        
        ## Move Planet 2 (influenced by Star and Planet 1)
        p2_new, v2_new = move_planets(pos2, vel2, m2, m_star, pos1, m1, dt)
        
        ## Update current state
        pos1, vel1 = p1_new, v1_new
        pos2, vel2 = p2_new, v2_new

        x1 , y1 = pos1
        x2,y2 = pos2
        
        ## Store results
        x1_arr.append(x1)
        y1_arr.append(y1)
        x2_arr.append(x2)
        y2_arr.append(y2)
        
        t += dt

    return np.array([x1_arr, y1_arr]), np.array([x2_arr, y2_arr])

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24
m_venus = 4.8e24

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

sub_earth = earth_traj[:, ::300]
sub_venus = venus_traj[:, ::300]

animations = ([sub_earth, sub_venus])
animate_trajectories_2d(animations,30)

As seen above the simulation appears to be working as intended. We can see that when Venus goes closer to the orbit of the Earth, the orbit of Venus speeds up until it moves on. This shows that the code is running correctly and that it is functioning asa intended. 

### Varying the mass of Venus

We can see how the orbit should like above when the mass of Venus is 4.8e24 Kg. lets see how it changes when we do

a) Equal to the Earth

b) Double the Earth

c) 0ver 300 times the Mass of the Earth

d) Over 500 times the Mass of the Earth

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24
m_venus = 6e24

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = earth_traj
x_venus , y_venus = venus_traj

# Plot trajectories
plt.figure()
plt.title('Orbits of Earth and Venus')
plot_trajectory(x_earth, y_earth)
plot_trajectory(x_venus, y_venus)

As we can see the orbit does not change much when the two masses are equal. This may be due to the fact that they wont have that much effect on the other due to the sun as that is already a much larger mass.


In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24
m_venus = 1.2e25

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = earth_traj
x_venus , y_venus = venus_traj

# Plot trajectories
plt.figure()
plt.title('Orbits of Earth and Venus')
plot_trajectory(x_earth, y_earth)
plot_trajectory(x_venus, y_venus)

As  seen from above there is still no deviation in the orbit of either Venus or Earth.

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24
m_venus = 6e24 * 300

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = earth_traj
x_venus , y_venus = venus_traj

# Plot trajectories
plt.figure()
plt.title('Orbits of Earth and Venus')
plot_trajectory(x_earth, y_earth)
plot_trajectory(x_venus, y_venus)

As we can see from above the orbit of the earth os beginnning to change from a purely circular orbit to an orbit which is beginning to deviate. 

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24
m_venus = 6e24 * 500

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

x_earth , y_earth = earth_traj
x_venus , y_venus = venus_traj

# Plot trajectories
plt.figure()
plt.title('Orbits of Earth and Venus')
plot_trajectory(x_earth, y_earth)
plot_trajectory(x_venus, y_venus)

We can now see the deviation in the orbit clearly as the mass of venus is now 500 times that of  the earth.

### Varying the Mass of Earth

For this we will alter the mass of Earth by multiplying it by 5000.

In [ ]:
## Calculate orbital
v_earth = np.sqrt(G*M_SOLAR/AU)

## Initial conditions of planet
pos_earth = np.array([AU, 0])       
vel_earth = np.array([0,v_earth])   

r_venus = 0.72 * AU
v_venus = np.sqrt(G*M_SOLAR/r_venus)
pos_venus = np.array([r_venus,0])     # initial position (x,y) [m]
vel_venus = np.array([0, v_venus])    # initial velocity (vx,vy) [m/s]

m_earth = 6e24 * 5000
m_venus = 4.8e24

## Calculate trajectory
dt = YEAR/25000 

earth_traj, venus_traj = trajectories(pos_earth,vel_earth,m_earth, pos_venus, vel_venus,m_venus,M_SOLAR,dt,5*YEAR)

sub_earth = earth_traj[:, ::300]
sub_venus = venus_traj[:, ::300]

animations = ([sub_earth, sub_venus])
animate_trajectories_2d(animations,30)

As seen above the orbit, the orbit drastically changes when the mass of the earth is multiplied by 5000. We can see that the orbit of venus begins to be pulled by the mass of the earth as it is closer to the earth than the sun. therefore there being a greater force of attraction between the two than that of the earth.

## Section 3: Further Investigation

For my investigation, I will be developing a solar systemm based on the data from our solar system. For this we will change the code that simulates two bodies being affected and extend that to be able make an N body system. This will involve for loops. Since the code above is optimised for a two planets interacting with each other. Therefore for multiple planets to be simulated I will need to use loops as well as arrays for it to work. I will be using values given from NASA [3].

In [ ]:
def move_further(positions, velocities, masses, m_star, dt):
    """
    Calculates the next state for N planets in the system.
    
    positions: List of arrays [pos1, pos2, ...]
    velocities: List of arrays [vel1, vel2, ...]
    masses: List of arrays [m1, m2, ...]
    """
    ## This is going to be used to determine the number of planets
    n = len(positions)
    ## These will be the new positions and velocities
    new_positions = []
    new_velocities = []
    star_pos = np.array([0.0, 0.0])

    for i in range(n):
        ## Force from the Star
        total_force = force(star_pos, positions[i], m_star, masses[i])
        
        ## Forces from all N planets
        for j in range(n):
            ## This condition will make sure that the planet isnt pulling itself 
            ## Ensuring that the loop does not break.
            if i == j: continue  
            total_force += force(positions[j], positions[i], masses[j], masses[i])

        accel = total_force / masses[i]    
        p_new = positions[i] + velocities[i] * dt
        v_new = velocities[i] + accel * dt
        
        new_positions.append(p_new)
        new_velocities.append(v_new)

    return new_positions, new_velocities

def trajectory_further(initial_positions, initial_velocities, masses, m_star, dt, t_max):
    """
    Simulates an entire solar system using Standard Euler with pre-allocation.
    """
    num_steps = int(t_max / dt)
    num_planets = len(initial_positions)

    ## This will be used to allocate sspace for all of the values of the trajectories 
    trajectories = np.zeros((num_planets, 2, num_steps))
    
    curr_pos = initial_positions
    curr_vel = initial_velocities

    for t in range(num_steps):
        for p in range(num_planets):
            trajectories[p, :, t] = curr_pos[p]
        
        curr_pos, curr_vel = move_further(curr_pos, curr_vel, masses, m_star, dt)

    return trajectories

In [ ]:
masses = [5.97e24, 1.90e27, 6.39e23] ## Earth, Jupiter, Mars
pos = [np.array([1.5e11, 0]), np.array([7.7e11, 0]), np.array([2.2e11, 0])]
vel = [np.array([0, 29780]), np.array([0, 13070]), np.array([0, 24000])]
dt = YEAR / 25000

all_trajectories = trajectory_further(pos, vel, masses, M_SOLAR, dt, 12*YEAR)

plt.figure(figsize=(10, 10))
planet_names = ["Earth", "Jupiter", "Mars"]

## Given that we have sent all the data as one block we will now need to split it up.

for i in range(len(all_trajectories)):
    ## Extract the data for the current planet i
    x_data = all_trajectories[i, 0, :]
    y_data = all_trajectories[i, 1, :]
    
    plot_trajectory(x_data, y_data, label=planet_names[i])
    
plt.title(f"Solar System Simulation (12 Years)")
plt.grid(True)
plt.legend()
plt.show()

As seen above I have been able to plot out the trajectories of three planets within the solar system; Earth, Mars and Jupiter. The time period that I had to use was 12 years so that we could see the complete trajectory of Jupiter fold out. We can also see that although only one pass has been made of Jupiter there have been multiple passes made of of both the Earth and Mars. we can tell this by the bold outlines of the trajectories of the planets. 

## Conclusion 

Within this notebook, I have simulated the orbits of Earth, Venus, Mars and Jupiter. First beginnning with simulating the Earth to see how the orbit is meant to work. I was then able to find which dt is best used to simulate a nice and smooth curve. Following that I had changed the linear velocity of the Earth at the start of its orbit to see how it would change the orbit. It started with an increasingly elliptical orbit when the starting velocity is half of the accepted velocity, followed by when the velocity is equal to the escape velocity we see that the Earth enters a hyperbolic trajectory to never return. 

This then moves on to the two bodies acting on each other and finding when the orbit of Earth changes when the mass of Venus increases. We had found that the orbit of Earth only changes when Venus is 500 times the mass of the original. It happens because there is a greater attraction than that of the sun due to the closeness of the two bodies. This is the same explanation as why the orbit of Venus changes when the mass of Earth increases. 

Foloowing this I had developed a peice of code for simulating a solar systems orbital map. Unfortunaltey for this I couldnt place all of the planets in this due to the extended time it would take. It was also unfeasible for the number of calculations that it would need to do.  

## References
1.SPHINX. Indexing on ndarrays — NumPy v1.22 Manual [Internet]. numpy.org. [cited 2026 Jan 21]. Available from: https://numpy.org/doc/stable/user/basics.indexing.html

2.The Editors of Encyclopaedia Britannica. Escape velocity | physics | Britannica. In: Encyclopædia Britannica [Internet]. 2020 [cited 2026 Jan 21]. Available from: https://www.britannica.com/science/escape-velocity

3.NASA. Solar System Sizes - NASA Science [Internet]. science.nasa.gov. 2023 [cited 2026 Jan 21]. Available from: https://science.nasa.gov/resource/solar-system-sizes/